# Convo AI: Phase 1 (text, Mandarin, ordering at a café)

An AI waiter (Qwen2.5-3B-Instruct) chats with an A1–A2 Mandarin learner, adapts to their level, and writes a feedback report afterwards. All code lives in the repo's `convo/` package; this notebook just runs it.

**Before running:** Runtime → Change runtime type → **T4 GPU** → Save. Then run the cells top to bottom.

| Section | What it does |
|---|---|
| 1 | Install, get the code, load the model |
| 2 | Quick smoke test |
| 3 | Evaluation on the **tune** sentences (use these while improving prompts) |
| 4 | Evaluation on the **test** sentences (**final score: don't tune after this**) |
| 5 | Conversation evaluation |
| 6 | Launch the app (for the demo video) |

## 1. Setup

In [ ]:
!pip -q install "transformers>=4.45" accelerate gradio pypinyin Pinyin2Hanzi

In [ ]:
import os, sys, shutil, time
REPO = "https://github.com/jackthescammer/Convo-ai-chatbot"
os.chdir("/content")
# Always start from a fresh copy so the newest prompt version is used.
# (A plain `git pull` can be silently refused when local result files clash with committed ones.)
if os.path.exists("Convo-ai-chatbot"):
    if os.path.isdir("Convo-ai-chatbot/results"):
        backup = f"/content/results_backup_{time.strftime('%H%M%S')}"
        shutil.copytree("Convo-ai-chatbot/results", backup)
        print("Backed up old local results to", backup)
    shutil.rmtree("Convo-ai-chatbot")
!git clone -q {REPO}
%cd /content/Convo-ai-chatbot
sys.path.insert(0, os.getcwd())
for m in [m for m in list(sys.modules) if m in ("convo", "app") or m.startswith("convo.")]:
    del sys.modules[m]   # forget any old version already imported
!git log --oneline -1
from convo.prompts import PROMPT_VERSION
print("PROMPT VERSION:", PROMPT_VERSION, "<-- check this is the version you were asked to run")

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU, then run again"

from convo.backends import load_backend      # change the model with load_backend("other/model-id")
backend = load_backend()                      # default: Qwen/Qwen2.5-3B-Instruct

## 2. Smoke test
One short conversation and a feedback report, to check everything is wired up.

In [ ]:
from convo import Settings
from convo.chat import Conversation
from convo.review import build_report

conv = Conversation(backend, Settings(level="A2", difficulty="Standard"), seed=1)
t = conv.start()
print("小林:", t.reply, "|", t.english, "| valid:", t.valid)
for line in ["你好！我要一个咖啡。", "热的，谢谢。"]:
    t = conv.send(line)
    print("你:", line, "\n小林:", t.reply, "|", t.english, "| valid:", t.valid, "| state:", t.stage, t.order)
print(build_report(backend, conv.learner_sentences, conv.settings).to_markdown())

## 3. Corrections: tune split (10 sentences)
Use this while improving the prompt in `convo/prompts.py`. **Bump `PROMPT_VERSION` each time you change a prompt**, then `!git pull` here (or restart the runtime) and re-run.

In [ ]:
from convo.evaluation import load_cases, run_correction_eval, log_result, format_metrics, export_for_human_review, RESULTS_DIR
cases = load_cases(split="tune")
res = run_correction_eval(backend, cases)
print(format_metrics(res["metrics"]))
log_result("corrections", backend.name, "tune", res)
import pandas as pd
pd.DataFrame(res["rows"])[["learner", "expected", "flagged", "model_fix", "category_model", "fix_exact"]]

## 4. Corrections: test split (final score)
⚠️ **Run this once, when the prompt is finished.** Changing the prompt after seeing these scores makes the result unfair.

Also compare against the baseline (a waiter that never corrects anything catches 0 mistakes and has 0 false corrections).

In [ ]:
cases = load_cases(split="test")
print(sum(1 for c in cases if not c["reviewed_by"].strip()), "of", len(cases), "sentences not yet checked by a Chinese speaker")
res_test = run_correction_eval(backend, cases)
print(format_metrics(res_test["metrics"]))
log_result("corrections", backend.name, "test", res_test)
export_for_human_review(res_test, RESULTS_DIR / "human_review_test.csv")
pd.DataFrame(res_test["metrics"]["by_category"]).T

In [ ]:
# Where it fails: show the model's answer next to the right one.
rows = pd.DataFrame(res_test["rows"])
fails = rows[(rows.has_mistake & ~rows.fix_exact) | (~rows.has_mistake & rows.flagged)]
for _, r in fails.head(6).iterrows():
    print("LEARNER :", r.learner, "\nEXPECTED:", r.expected or "(already correct)", "\nMODEL   :", r.model_fix or "(no correction)", "\nWHY     :", r.explanation_model, "\n")

### Human review
Strict matching counts a correction worded differently from ours as wrong, so the numbers above are a **lower bound**.
1. Download `results/human_review_test.csv` (Files panel on the left).
2. A Chinese speaker fills in `human_correction_ok` and `human_explanation_ok` with `y` or `n`.
3. Upload it again and run the next cell.

In [ ]:
from convo.evaluation import summarise_human_review
summarise_human_review("results/human_review_test.csv")

## 5. Conversation evaluation
Scripted learner turns; the waiter's replies are always generated. Checks character consistency, context retention and order tracking.

In [ ]:
from convo.evaluation import run_conversation_eval
conv_res = run_conversation_eval(backend, Settings(level="A2", difficulty="Standard"))
print(format_metrics(conv_res["metrics"]))
log_result("conversation", backend.name, None, conv_res)
for t in conv_res["transcripts"][:2]:
    print("\n--", t["script"], "| context retained:", t["context_retained"])
    for d in t["dialogue"]:
        print("  你:", d["learner"], "\n  小林:", d["waiter"])

### Save your results
Colab files vanish when the runtime resets. Download `results/` (it records every model and prompt version) and commit it to the repo.

In [ ]:
from convo.prompts import PROMPT_VERSION
!rm -f results.zip
!zip -qr results.zip results
from google.colab import files
print("Downloading results for prompt version", PROMPT_VERSION)
files.download("results.zip")

## 6. Launch the app
Onboarding → conversation → report.

**Use 6b for the website.** It serves the app at the same fixed address every time (ngrok), so tryconvo.vercel.app keeps working after a restart. It needs two Colab Secrets (🔑 key icon on the left, turn on "Notebook access" for both):
- `NGROK_AUTHTOKEN`: from dashboard.ngrok.com → Your Authtoken
- `NGROK_DOMAIN`: your free static domain from dashboard.ngrok.com → Domains (e.g. `something.ngrok-free.app`)

6a below is the old way: a temporary gradio.live link that changes every launch. Run **only one** of 6a or 6b.

In [ ]:
# 6a. Temporary link (changes every launch)
from app import build_app
build_app(backend).launch(share=True, debug=True)

In [ ]:
# 6b. Fixed link for the website (same address every launch)
!pip -q install ngrok
import ngrok
from google.colab import userdata
from app import build_app

token, domain = userdata.get("NGROK_AUTHTOKEN"), userdata.get("NGROK_DOMAIN")
assert token and domain, "Add NGROK_AUTHTOKEN and NGROK_DOMAIN in Colab Secrets (key icon on the left) first"
try:
    await ngrok.disconnect()  # close a tunnel left over from an earlier run of this cell
except Exception:
    pass
listener = await ngrok.forward(7860, authtoken=token, domain=domain)  # in a notebook, ngrok calls must be awaited
print("Live at", listener.url(), "<- the website's model link")
build_app(backend).launch(server_name="127.0.0.1", server_port=7860, debug=True)